# 02b — Websach Cleaning Pipeline

Notebook này chạy pipeline reproducible để biến Websach laptop data thành clean dataset sẵn sàng cho feature engineering/modeling.

Logic cleaning nằm trong `src/clean/websach_cleaning.py`; notebook này chỉ import module, chạy pipeline, và hiển thị audit summary.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != "Laptop-Price-Prediction":
    candidates = [p for p in PROJECT_ROOT.parents if p.name == "Laptop-Price-Prediction"]
    PROJECT_ROOT = candidates[0] if candidates else PROJECT_ROOT

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.clean.websach_cleaning import CONFIG, run_pipeline

CONFIG

{'raw_path': WindowsPath('Y:/Python/Laptop-Price-Prediction/data/raw/clean_laptop_features.csv'),
 'issues_path': WindowsPath('Y:/Python/Laptop-Price-Prediction/docs/websach_issues_list.csv'),
 'output_dir': WindowsPath('Y:/Python/Laptop-Price-Prediction/data/intern'),
 'cleaned_path': WindowsPath('Y:/Python/Laptop-Price-Prediction/data/intern/websach_cleaned.csv'),
 'report_path': WindowsPath('Y:/Python/Laptop-Price-Prediction/docs/websach_cleaning_report.csv'),
 'issue_action_plan_path': WindowsPath('Y:/Python/Laptop-Price-Prediction/docs/websach_issue_action_plan.csv'),
 'log_path': WindowsPath('Y:/Python/Laptop-Price-Prediction/docs/websach_cleaning_log.json'),
 'price_cols': ['shop_1_price', 'shop_2_price', 'shop_3_price'],
 'shop_name_cols': ['shop_1_name', 'shop_2_name', 'shop_3_name'],
 'price_min': 3000000,
 'price_max': 200000000,
 'relative_low_factor': 0.5,
 'relative_high_factor': 2.0,
 'price_spread_warn': 0.3,
 'price_spread_critical': 0.5,
 'screen_min': 0,
 'screen_max

## Run Pipeline

Các bước trong `run_pipeline`: load clean Websach input + issue list, build `issue_action_plan`, filter price cells, compute `price_median`, normalize specs into tiers/groups, flag quality issues, validate, and save outputs.

In [2]:
artifacts = run_pipeline(CONFIG)

Raw schema:
Hãng sản xuất              object
Công nghệ CPU              object
Loại CPU                   object
Dung lượng RAM            float64
Loại RAM                   object
Loại ổ cứng                object
Dung lượng ổ cứng (GB)    float64
Đồ họa đã làm sạch         object
Kích thước (inch)         float64
shop_1_price                int64
shop_1_name                object
shop_2_price                int64
shop_2_name                object
shop_3_price              float64
shop_3_name                object
dtype: object

Raw shape: 4,384 rows x 15 columns

Sample records:
  Hãng sản xuất       Công nghệ CPU Loại CPU  Dung lượng RAM Loại RAM Loại ổ cứng  Dung lượng ổ cứng (GB)  Đồ họa đã làm sạch  Kích thước (inch)  shop_1_price           shop_1_name  shop_2_price        shop_2_name  shop_3_price      shop_3_name
0          Acer       Intel Core i5    8265U             8.0     DDR4         SSD                   512.0  Intel UHD Graphics               14.0      14950000        

## Quick Audit

In [3]:
print('Cleaned shape:', artifacts.cleaned.shape)
print('\nIssue status counts:')
print(artifacts.issue_action_plan['resolution_status'].value_counts().to_string())
print('\nImportant rule counts:')
for key, value in artifacts.log['important_rule_counts'].items():
    print(f'- {key}: {value}')

artifacts.cleaned.head()

Cleaned shape: (4384, 67)

Issue status counts:
resolution_status
needs_manual_review    34
resolved               12
partially_resolved     11

Important rule counts:
- domain_price_cells_nulled: 4
- relative_price_cells_nulled: 53
- price_spread_warn_rows: 583
- price_spread_critical_rows: 142
- screen_size_outlier_nulled: 85
- ram_capacity_outlier_nulled: 2
- soft_duplicate_spec_rows: 2516


,Hãng sản xuất,Công nghệ CPU,Loại CPU,Dung lượng RAM,Loại RAM,Loại ổ cứng,Dung lượng ổ cứng (GB),Đồ họa đã làm sạch,Kích thước (inch),shop_1_price,...,product_id,spec_key,is_soft_duplicate_spec,high_end_config_flag,brand_screen_cell_n,low_count_brand_screen_cell,cpu_ram_cell_n,gpu_ram_cell_n,low_count_cpu_ram_cell,low_count_gpu_ram_cell
0,Acer,Intel Core i5,8265U,8.0,DDR4,SSD,512.0,Intel UHD Graphics,14.0,14950000,...,0,Acer|8265U|8.0|512.0|14.0,False,False,84,False,1774,1101,False,False
1,Lg,Intel Core i7,1165G7,16.0,LPDDR4x,SSD,512.0,Intel Iris Xe,17.0,32640000,...,1,LG|1165G7|16.0|512.0|17.0,False,False,6,True,1297,753,False,False
2,Lenovo,AMD Ryzen 7,8845HS,16.0,DDR5,SSD,512.0,RTX 4060,16.0,30900000,...,2,Lenovo|8845HS|16.0|512.0|16.0,False,True,184,False,252,180,False,False
3,Lenovo,AMD Ryzen 7,7840H,16.0,LPDDR5,SSD,512.0,RTX4060,15.6,23900000,...,3,Lenovo|7840H|16.0|512.0|15.6,True,True,170,False,252,180,False,False
4,Msi,Intel Core Ultra 7,255HX,16.0,DDR5,SSD,512.0,RTX 5070,16.0,54790000,...,4,MSI|255HX|16.0|512.0|16.0,False,True,39,False,1297,56,False,False


In [4]:
artifacts.issue_action_plan.head(15)

,issue_key,severity,affected_column,issue_group,issue_description,proposed_action,need_note_lookup,final_action,resolution_status
0,price_slots_not_fixed_shops,low,shop_*_price/price_median,price_related_issues,"Các cột shop_1_price, shop_2_price, shop_3_pri...",Diễn giải các cột này là price slots; chuyển d...,no,Documented; no deterministic cleaning required.,needs_manual_review
1,price_domain_outliers,high,shop_*_price/price_median,price_related_issues,Tồn tại một số giá trị giá nằm ngoài miền hợp ...,Áp dụng domain filter ở cấp độ từng ô giá: nul...,yes,Apply cell/field-level null-out or flag; do no...,resolved
2,price_relative_outliers,high,shop_*_price/price_median,price_related_issues,Một số sản phẩm có một nguồn giá lệch mạnh so ...,"Sau domain filter, dùng median giá của từng dò...",yes,Apply cell/field-level null-out or flag; do no...,resolved
3,price_spread_between_sources,high,shop_*_price/price_median,price_related_issues,"Sau khi lọc domain outlier, vẫn có nhiều sản p...",Không xóa toàn bộ các dòng có spread cao; tạo ...,no,Documented; no deterministic cleaning required.,needs_manual_review
4,price_right_skewed_distribution,low,shop_*_price/price_median,price_related_issues,Giá tham chiếu price_median có phân phối lệch ...,Dùng median/IQR để mô tả giá thay vì chỉ dùng ...,no,Document as modeling guidance; keep engineered...,needs_manual_review
5,price_high_tail_valid_but_influential,low,shop_*_price/price_median,price_related_issues,Nhóm giá cao sau cleaning phần lớn có cấu hình...,Giữ lại nếu mục tiêu là bao phủ toàn bộ thị tr...,yes,Document as modeling guidance; keep engineered...,needs_manual_review
6,source_quality_variation,high,,price_related_issues,Một số shop_name có tỷ lệ relative outlier cao...,Không loại bỏ toàn bộ shop chỉ dựa trên tỷ lệ ...,no,Apply cell/field-level null-out or flag; do no...,resolved
7,screen_size_missing_values,high,Kích thước (inch),price_related_issues,Cột Kích thước (inch) có một số giá trị thiếu....,Không nên xóa dòng chỉ vì thiếu kích thước màn...,yes,Document as modeling guidance; keep engineered...,needs_manual_review
8,screen_size_outliers,high,Kích thước (inch),price_related_issues,Tồn tại các giá trị kích thước màn hình lớn hơ...,Null-out các giá trị Kích thước (inch) > 25 ho...,yes,Apply cell/field-level null-out or flag; do no...,resolved
9,screen_size_fragmentation,high,Kích thước (inch),price_related_issues,Kích thước màn hình có nhiều giá trị chi tiết ...,Gom kích thước màn hình thành các nhóm như '<1...,yes,Keep Unknown/tier flags instead of dropping rows.,partially_resolved


In [5]:
artifacts.report.head(30)

,section,metric,value
0,shape,row_count_before,4384
1,shape,row_count_after,4384
2,shape,column_count_before,15
3,shape,column_count_after,67
4,duplicates,full_duplicate_removed,0
5,duplicates,soft_duplicate_spec_rows,2516
6,price,domain_price_cells_nulled,4
7,price,relative_price_cells_nulled,53
8,price,price_spread_warn_rows,583
9,price,price_spread_critical_rows,142
